## Load Document

In [1]:
import os
import asyncio
from pathlib import Path
import json
from dotenv import load_dotenv
from langchain_core.documents import Document
from typing import List
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

In [2]:
path = Path("../data/contracts")
pdf_paths = list(path.glob("*.pdf"))
processed_pdf = Path("../data/processed_contracts")
pdf_paths, processed_pdf

([WindowsPath('../data/contracts/800263424202323950PMSLA-3208053.pdf'),
  WindowsPath('../data/contracts/OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).pdf')],
 WindowsPath('../data/processed_contracts'))

In [ ]:
async def main():
    if not pdf_paths:
        raise ValueError("No PDF files found.")
    
    client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

    for pdf_path in pdf_paths:
        print(f"Processing {pdf_path.name}")

        with open(pdf_path, "rb") as f:
            file_obj = await client.files.create(file=f, purpose="parse")

        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="agentic_plus",
            version="latest",
            output_options={
                "markdown": {
                    "tables": {
                        "output_tables_as_markdown": True
                    }
                }
            },
            expand=["markdown"],
        )

        print(f"Result type: {type(result)}")
        print(f"Markdown: {result.markdown}")
        print(f"Result keys: {result.__dict__.keys()}")

        if result.markdown is None or result.markdown.pages is None:
            print(f"Warning: No markdown returned for {pdf_path.name}, trying text fallback...")
            
            markdown_result = await client.parsing.get_job_markdown(job_id=result.id)
            full_text = markdown_result if isinstance(markdown_result, str) else str(markdown_result)
        else:
            full_text = "\n\n".join(page.markdown for page in result.markdown.pages)

        output_file = processed_pdf / f"{pdf_path.stem}_parsed.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(full_text)

        print(f"Saved: {output_file}")

await main()

In [3]:
from pydantic import BaseModel, Field
from typing import Optional


class SupplierInfo(BaseModel):
    service_provider_name: Optional[str] = None
    client_name: Optional[str] = None
    agreement_date: Optional[str] = None
    contract_duration: Optional[str] = None
    model_config = {"extra": "forbid"}

class UptimeCommitments(BaseModel):
    guaranteed_uptime_percent: Optional[float] = None
    measurement_period: Optional[str] = None
    excluded_downtime: Optional[str] = None
    model_config = {"extra": "forbid"}

class PenaltyClause(BaseModel):
    clause_id: Optional[str] = None
    trigger_condition: Optional[str] = None
    penalty_type: Optional[str] = None
    penalty_amount: Optional[str] = None
    model_config = {"extra": "forbid"}

class TerminationClauses(BaseModel):
    termination_for_cause: Optional[str] = None
    notice_period_days: Optional[float] = None
    termination_for_convenience: Optional[str] = None
    model_config = {"extra": "forbid"}

class ForceMajeure(BaseModel):
    clause_id: Optional[str] = None
    covered_events: Optional[str] = None
    notification_requirement: Optional[str] = None
    model_config = {"extra": "forbid"}

class DisputeResolution(BaseModel):
    governing_law: Optional[str] = None
    resolution_mechanism: Optional[str] = None
    model_config = {"extra": "forbid"}

class SLADocument(BaseModel):
    supplier_info: Optional[SupplierInfo] = None
    uptime_commitments: Optional[UptimeCommitments] = None
    penalty_clauses: Optional[list[PenaltyClause]] = None
    termination_clauses: Optional[TerminationClauses] = None
    force_majeure: Optional[ForceMajeure] = None
    dispute_resolution: Optional[DisputeResolution] = None
    model_config = {"extra": "forbid"}

In [4]:
schema = SLADocument.model_json_schema()

from llama_cloud import LlamaCloud

client = LlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

In [9]:
for pdf in pdf_paths:
    print(f"Processing: {pdf.name}")
    with open(pdf, "rb") as f:
        file_obj = client.files.create(file=(pdf.name, f, "application/pdf"), purpose="extract")

    result = client.extraction.extract(
        file_id=file_obj.id,
        data_schema=schema,
        config={},
    )

    # Debug: see what actually came back
    print("Raw result:", result.data)

    sla = SLADocument(**result.data)

    output_path = processed_pdf / f"{pdf.stem}.json"
    output_path.write_text(sla.model_dump_json(indent=2, exclude_none=True), encoding="utf-8")
    print(f"{pdf.name} → {output_path}")

    # Safe access
    if sla.supplier_info:
        print(f"  Provider : {sla.supplier_info.service_provider_name}")
    if sla.uptime_commitments:
        print(f"  Uptime   : {sla.uptime_commitments.guaranteed_uptime_percent}%")
    else:
        print("uptime_commitments not found in document")

Processing: 800263424202323950PMSLA-3208053.pdf
Raw result: {'supplier_info': {'service_provider_name': 'Mazagon Dock Shipbuilders Limited (MDL)', 'client_name': 'Mazagon Dock Shipbuilders Limited', 'agreement_date': None, 'contract_duration': 'two years from the date of placement of Order'}, 'uptime_commitments': None, 'penalty_clauses': None, 'termination_clauses': {'termination_for_cause': 'If the equipment / article / service or any portion thereof be not delivered/ performed by the scheduled delivery date/ period, any stoppage or discontinuation of ordered supply / awarded contract without written consent by Purchaser or not meeting the required quality standards the Purchaser shall be at liberty, without prejudice to the right of the Purchaser to recover Liquidated Damages / penalty as provided for in these conditions or to any other remedy for breach of contract, to terminate the contract either wholly or to the extent of such default.', 'notice_period_days': None, 'termination_

## Chunking

In [5]:
from langchain_community.document_loaders import TextLoader

e:\SentryChain-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 2000, chunk_overlap = 500):
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\nClause", "\nSection", " ", ""],
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Splitting {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content {split_docs[0].page_content[:100]}...")

    return split_docs

### Metadata:
* supplier
* section_id
* category(force majeure, penalty, termination)

In [7]:
processed_json_contracts = Path("../data/processed_contracts")
processed_json_contracts_paths = list(processed_json_contracts.glob("*.json"))
processed_json_contracts_paths

[WindowsPath('../data/processed_contracts/800263424202323950PMSLA-3208053.json'),
 WindowsPath('../data/processed_contracts/OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).json')]

In [8]:
from langchain_community.graphs import Neo4jGraph

In [10]:
from sentence_transformers import SentenceTransformer

In [11]:
class EmbeddingManager:
    def __init__(self, model_name: str = "BAAI/bge-m3" ):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        print(f"Loading model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


    def generate_embeddings(self, texts: List[str]):
        embeddings = self.model.encode(texts, normalize_embeddings=True)
        if len(texts) == 1:
            return embeddings[0].astype(float).tolist()  
        return embeddings.astype(float).tolist()    

embedding_manager = EmbeddingManager()

Loading model: BAAI/bge-m3
Model loaded successfully. Embedding dimension: 1024


In [12]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name= "sentry-chain-ai"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Manual index '{index_name}' created.")

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),
    database='neo4j'
)

dense_index = pc.Index(index_name)

C:\Users\GOURAV\AppData\Local\Temp\ipykernel_14836\1331176472.py:15: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [13]:
def data_ingestion(sla_data, contract_id, processed_pdf_path, embedding_manager, dense_index, graph):

    # load supplier name
    supplier_name = sla_data.supplier_info.service_provider_name if sla_data.supplier_info else "Unknown"

    text_file = processed_pdf_path / f"{contract_id}_parsed.txt"
    loader = TextLoader(file_path=str(text_file), encoding="utf-8")
    docs = loader.load()

    contract_chunks = split_docs(documents=docs)

    print("Embedding")
    embedding = [chunk.page_content for chunk in contract_chunks]
    batch_embedding = embedding_manager.generate_embeddings(embedding)

    supplier_name = sla_data.supplier_info.service_provider_name if sla_data.supplier_info else "Unknown"

    # supplier -> contract
    graph.query("""
    MERGE (s:Supplier {name: $s_name})
    MERGE (c:Contract {id: $c_id})
    MERGE (s)-[:HAS_CONTRACT]->(c)
    SET c.last_updated = timestamp()
    """, params={
        "s_name": supplier_name,
        "c_id": contract_id
    })

    # Penalty Nodes for Graph Reasoning
    if sla_data.penalty_clauses:
        for idx, p in enumerate(sla_data.penalty_clauses):
            graph.query("""
            MATCH (c:Contract {id: $c_id})
            MERGE (pc:PenaltyRule {id: $p_uid})
            SET pc.type = $type, 
                pc.trigger = $trigger, 
                pc.amount = $amount,
                pc.clause_ref = $ref
            MERGE (c)-[:ENFORCES_PENALTY]->(pc)
            """, params={
                "c_id": contract_id,
                "p_uid": f"rule_{contract_id}_{idx}",
                "type": p.penalty_type,
                "trigger": p.trigger_condition,
                "amount": p.penalty_amount,
                "ref": p.clause_id
            })


    print(f"Upserting {len(contract_chunks)} chunks to Pinecone...")
    for i, (chunk, emb) in enumerate(zip(contract_chunks, batch_embedding)):
        chunk_id = f"{contract_id}#chunk{i}"

        graph.query("""
        MATCH (c:Contract {id: $c_id})
        MERGE (ch:Chunk {vector_id: $v_id})
        SET ch.text_preview = $text
        MERGE (c)-[:HAS_CONTENT_CHUNK]->(ch)
        """, params={
            "c_id": contract_id,
            "v_id": chunk_id,
            "text": chunk.page_content[:200] # Storing preview for graph visualization
        })

        vector_values = embedding_manager.generate_embeddings([chunk.page_content])

        dense_index.upsert(vectors=[{
            "id": chunk_id,
            "values": emb,
            "metadata": {
                **chunk.metadata,
                "text": chunk.page_content,
                "supplier_name": supplier_name,
            }
        }])

In [14]:
for pdf_path in pdf_paths:
    contract_id = pdf_path.stem

    json_path = processed_json_contracts / f"{contract_id}.json"

    with open(json_path, 'r', encoding="utf-8") as f:
        sla_dict = json.load(f)
    
    sla = SLADocument(**sla_dict)

    data_ingestion(sla_data=sla,
                    contract_id=contract_id,
                    processed_pdf_path= processed_pdf,
                    embedding_manager=embedding_manager,
                    dense_index=dense_index,
                    graph=graph)

Splitting 1 documents into 163 chunks
Content # SERVICE LEVEL AGREEMENT (SLA)

Mazagon Dock Shipbuilders Limited invites on-line competitive bids ...
Embedding


[#F5D3]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.9285374452794637s (Unable to retrieve routing information)
[#F5D4]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Transaction failed and will be retried in 1.939486027575205s (Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Upserting 163 chunks to Pinecone...
Splitting 1 documents into 278 chunks
Content # Service Level Agreement for Microsoft Online Services
# January 1, 2026

# Table of Contents

TABL...
Embedding


[#C4EE]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.8239026353703297s (Unable to retrieve routing information)
[#C4EC]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): OSError('No data')
Transaction failed and will be retried in 1.7694986268746045s (Failed to read from defunct connection IPv4Address(('si-04d2c4a4-8d79.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Upserting 278 chunks to Pinecone...


In [ ]:
# for i, chunk in enumerate(chunks):
#     const_id = f"{chunk.metadata['supplier']}#{i}"
#     supplier_name = chunk.metadata.get("supplier_name")

#     # push to neo4j
#     graph.query("""
#     MERGE (s:Supplier {name: $supplier})
#     MERGE (c:Contract {id: $contract_id})
#     SET c.source = $source
#     MERGE (ch:Chunk {vector_id: $v_id})
#     MERGE (s)-[:HAS_CONTRACT]->(c)
#     MERGE (c)-[:HAS_DATA]->(ch)
#     SET ch.text_preview = $text
#     """, params={
#         "supplier": supplier_name,
#         "contract_id": chunk.metadata['supplier_id'],
#         "v_id": const_id,
#         "text": chunk.page_content,
#         "source": chunk.metadata['source']
#     })

#     # push to pinecone
#     vector_values = embedding_manager.generate_embeddings([chunk.page_content])
#     dense_index.upsert(vectors=[{
#         "id": const_id,
#         "values": vector_values,
#         "metadata": {**chunk.metadata, "text": chunk.page_content}
#     }])

In [15]:
from langchain_groq import ChatGroq

groq_llm = ChatGroq(api_key=os.getenv('GROQ_API_KEY'), model="llama-3.3-70b-versatile", temperature=0.1)

In [16]:
def retriever(query:str, supplier_name: str):
    # Vector search
    query_embedding = embedding_manager.generate_embeddings([query])
    vector_db_result = dense_index.query(
        vector=query_embedding,
        top_k=5,
        include_metadata=True,
        filter={"supplier_name": {"$eq": supplier_name}}
    )

    retrieved_vector_ids = [m['id'] for m in vector_db_result['matches']]

    # Graph search
    graph_context = graph.query("""
    MATCH(s:Supplier{name:$supplier_name})-[:HAS_CONTRACT]->(c)-[:HAS_CONTENT_CHUNK]->(ch)
    WHERE ch.vector_id IN $vector_ids  
    RETURN ch.vector_id as id, ch.text_preview as preview
    """, params={
        "supplier_name": supplier_name,  
        "vector_ids": retrieved_vector_ids 
    })

    confirmed_ids = {item['id'] for item in graph_context}

    verified_results = [m for m in vector_db_result['matches'] if m['id'] in confirmed_ids]

    return graph_context, vector_db_result, verified_results

In [17]:
from langchain_core.prompts import PromptTemplate 

In [18]:
## For testing

def rag_query(query:str, contract_id:str):

    json_path = processed_json_contracts / f"{contract_id.replace('_parsed', '')}.json"
    with open(json_path, 'r') as f:
        data = json.load(f)
        supplier_name = data.get("supplier_info", {}).get("service_provider_name", "Unknown")

    graph_context, _, verified_results = retriever(query=query, supplier_name=supplier_name)

    vector_texts = [match['metadata']['text'] for match in verified_results]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))

    prompt = PromptTemplate(
        template="""You are a contract analyst. Answer the question based ONLY on the provided context.

            Context from {supplier_name}'s SLA:
            {context}

            Question: {question}
            Answer (cite specific clauses when possible):""",
                    input_variables=["supplier_name", "context", "question"]
        )

    formatted_prompt = prompt.format(
        supplier_name=supplier_name,
        context="\n\n".join(combined_contexts),
        question=query
    )

    response = groq_llm.invoke(formatted_prompt)

    return {
        "answer": response.content,
        "sources": [m['id'] for m in verified_results[:3]],
        "context_used": combined_contexts,
        "display_name": supplier_name
    }


result = rag_query(
    query="What are the penalty clauses for service downtime?",
    contract_id="OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed"
)

print(result['answer'])
print(f"\nSources: {result['sources']}")
print(f"\n {result['display_name']}")

The penalty clauses for service downtime are in the form of Service Credits, which are outlined in the Service Credit sections for each service. 

For example, in **Microsoft Cloud App Security**, if the Uptime Percentage is:
- Less than 99.9%, the Service Credit is 10% (as per the table: | Uptime Percentage | Service Credit | | ----------------- | -------------- | | < 99.9%           | 10%            |).
- Less than 99%, the Service Credit is 25% (as per the table: | Uptime Percentage | Service Credit | | ----------------- | -------------- | | < 99%             | 25%            |).

Similarly, for other services like **Microsoft Kaizala Pro**, **Microsoft Power Apps**, **Dynamics 365**, and **Dynamics 365 Guides**, the Service Credits are as follows:
- **Microsoft Kaizala Pro**: 
  - Less than 99.9%, the Service Credit is 25%.
  - Less than 99%, the Service Credit is 50%.
  - Less than 95%, the Service Credit is 100%.
- **Microsoft Power Apps**: 
  - Less than 99.9%, the Service Credi

## Phase 2 News Fetching

In [19]:
target_contract_id = "OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed"

In [20]:
json_path = processed_json_contracts / f"{target_contract_id.replace('_parsed', '')}.json"
with open(json_path, 'r') as f:
    data = json.load(f)
    supplier_name = data.get("supplier_info", {}).get("service_provider_name", "Unknown")

In [21]:
from langchain_tavily import TavilySearch, TavilyExtract

def fetch_news(supplier_name, score_threshold:float = 0.5, current_year = "2026"):
    tavily = TavilySearch(api_key=os.getenv('TAVILY_API_KEY'),
                        search_depth="advanced",
                        max_results=5)

    query = f"{supplier_name} supply chain disruption OR outage OR compliance risk OR SLA breach {current_year}"

    results = tavily.invoke(query)

    filtered_results = [r for r in results['results'] if r['score'] >= score_threshold]
    return filtered_results

In [22]:
supplier_news_result = fetch_news(supplier_name=supplier_name)

In [23]:
supplier_news_result

[{'url': 'https://www.linkedin.com/posts/mohammed-rafi-710308263_recent-outages-february-2026-microsoft-activity-7427592066135793665-P67Q',
  'title': 'Recent Outages (February 2026) Microsoft 365 Admin Center (Feb ...',
  'content': '⚡ Microsoft Data Center Power Outage Disrupts Windows 11 Updates and Store Functionality | Source:  Microsoft has confirmed that a significant power outage at one of its West US data centers triggered widespread service disruptions yesterday, leaving thousands of Windows 11 users unable to access the Microsoft Store or complete Windows Updates. The incident, which began early Saturday morning, highlights the fragility of centralized cloud infrastructure even amidst robust redundancy protocols. The disruption began at approximately 08:00 UTC on February 7, 2026. Users across multiple regions, but specifically those relying on Azure services hosted in the West US region, began reporting timeouts and failures when attempting to download applications or fetch

In [24]:
def compare_sla(supplier_name : str, news_results : list):
    news_content = "".join([f"Source:{article['title']}\n{article['content']}" for article in news_results])

    query = "service outage uptime penalty SLA breach downtime"
    graph_context, vector_result, verified_results = retriever(query=query , supplier_name=supplier_name)
    vector_texts = [match['metadata']['text'] for match in verified_results]
    graph_texts = [item['preview'] for item in graph_context]

    combined_contexts = list(set(vector_texts+graph_texts))
    prompt = f"""You are a contract risk analyst. 

        NEWS EVENTS:
        {news_content}

        SLA CLAUSES:
        {combined_contexts}

        Based on the news events above, analyze:
        1. Has an SLA violation likely occurred? (Yes/No/Unclear)
        2. Which specific clauses are triggered?
        3. What penalties or remedies apply?
        4. Overall risk severity? (Low/Medium/High/Critical)

        Be specific and cite clause numbers where possible."""

    response = groq_llm.invoke(prompt)
    return {
        "supplier_id": supplier_name,
        "verdict": response.content,
        "news_used": [a['title'] for a in news_results],
        "sla_clauses_matched": combined_contexts
    }

In [25]:
compare_sla(supplier_name=supplier_name, news_results=supplier_news_result)

{'supplier_id': 'Microsoft',
 'verdict': 'Based on the news events, I analyze the situation as follows:\n\n1. **Yes**, an SLA violation has likely occurred. The news events report multiple outages and disruptions to Microsoft services, including Microsoft 365, Azure, and Windows 11 updates, which suggests that the services have not met the uptime and availability standards specified in the SLA.\n\n2. The specific clauses that are triggered include:\n\t* **Downtime** clauses, such as those found in the Exchange Online, Microsoft Teams, and Windows 365 sections, which define downtime as any period of time when users are unable to access or use the services.\n\t* **Uptime Percentage** clauses, such as those found in the Exchange Online, Microsoft Teams, and Windows 365 sections, which calculate the uptime percentage based on the formula: (User Minutes - Downtime) / User Minutes * 100.\n\t* **Service Credit** clauses, such as those found in the Exchange Online, Microsoft Teams, and Windows